In [7]:
from pathlib import Path
import pandas as pd
import re
from tqdm import tqdm

# --- Percorsi progetto (coerenti con i tuoi notebook) ---
METADATA_PATH = Path("/work/data/metadata/metadata_final.csv")

SUBS_RAW_DIR   = Path("subtitles_raw")     # .srt originali
SUBS_CLEAN_DIR = Path("subtitles_clean")   # .txt puliti (da sovrascrivere)
LOGS_DIR       = Path("logs")

SUBS_CLEAN_DIR.mkdir(exist_ok=True)
LOGS_DIR.mkdir(exist_ok=True)

metadata = pd.read_csv(METADATA_PATH)
metadata["imdb_id"] = metadata["imdb_id"].astype(str).str.strip()

print("metadata shape:", metadata.shape)
print("SRT disponibili:", len(list(SUBS_RAW_DIR.glob("*.srt"))))
print("TXT puliti attuali:", len(list(SUBS_CLEAN_DIR.glob("*.txt"))))


metadata shape: (4100, 9)
SRT disponibili: 1
TXT puliti attuali: 1


In [20]:
# --- Regex SRT ---
TIMESTAMP_RE = re.compile(r"\d{2}:\d{2}:\d{2},\d{3}\s*-->\s*\d{2}:\d{2}:\d{2},\d{3}")
INDEX_RE     = re.compile(r"^\d+$")
TAG_RE       = re.compile(r"<[^>]+>")

# bracket paratestuali comuni: [music], [applause], (laughs), etc.
# NB: NON rimuove parentesi "normali" dentro frasi (es. "he said (quietly)").
PAREN_ANNOT_RE  = re.compile(r"^\s*\((?:[^)]{1,40})\)\s*$")
BRACK_ANNOT_RE  = re.compile(r"^\s*\[(?:[^\]]{1,40})\]\s*$")

def clean_srt_to_text(raw: str) -> str:
    kept = []
    for line in raw.splitlines():
        s = line.strip()
        if not s:
            continue
        if INDEX_RE.match(s):
            continue
        if TIMESTAMP_RE.search(s):
            continue

        s = TAG_RE.sub("", s).strip()

        # elimina SOLO righe interamente "annotazione"
        if PAREN_ANNOT_RE.match(s) or BRACK_ANNOT_RE.match(s):
            continue

        # normalizza trattini iniziali tipici del dialogo
        s = re.sub(r"^\-\s*", "", s)

        # comprime spazi multipli
        s = re.sub(r"\s+", " ", s).strip()

        if s:
            kept.append(s)

    # separa le battute con newline (utile per LLM)
    return "\n".join(kept).strip()


In [3]:
import shutil
from datetime import datetime

BACKUP_DIR = Path("subtitles_clean_backup") / datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

existing_txt = list(SUBS_CLEAN_DIR.glob("*.txt"))
for p in existing_txt:
    shutil.copy2(p, BACKUP_DIR / p.name)

print("Backup creato in:", BACKUP_DIR)
print("File backuppati:", len(existing_txt))


Backup creato in: subtitles_clean_backup/20260214_135725
File backuppati: 3788


In [7]:
!pip install tqdm
from tqdm import tqdm



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [13]:
import re
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# Percorsi
METADATA_PATH = Path("/work/data/metadata/metadata_final.csv")
SUBS_RAW_DIR = Path("subtitles_raw")
SUBS_CLEAN_DIR = Path("subtitles_clean")
LOGS_DIR = Path("logs")

SUBS_CLEAN_DIR.mkdir(exist_ok=True)
LOGS_DIR.mkdir(exist_ok=True)

# Carico metadata
metadata = pd.read_csv(METADATA_PATH)
metadata["imdb_id"] = metadata["imdb_id"].astype(str).str.strip()

print("Film in metadata:", len(metadata))


Film in metadata: 4100


In [21]:
report = []

for imdb_id in tqdm(metadata["imdb_id"].unique(), desc="Cleaning+overwrite"):
    srt_path = SUBS_RAW_DIR / f"{imdb_id}.srt"
    out_path = SUBS_CLEAN_DIR / f"{imdb_id}.txt"

    if not srt_path.exists():
        report.append({
            "imdb_id": imdb_id,
            "status": "missing_srt",
            "raw_chars": 0,
            "clean_chars": 0,
            "raw_words": 0,
            "clean_words": 0,
            "shrink_ratio": None
        })
        continue

    raw = srt_path.read_text(encoding="utf-8", errors="ignore")
    clean = clean_srt_to_text(raw)

    # overwrite del TXT pulito
    out_path.write_text(clean, encoding="utf-8")

    raw_words = len(raw.split())
    clean_words = len(clean.split())
    raw_chars = len(raw)
    clean_chars = len(clean)

    shrink_ratio = (clean_words / raw_words) if raw_words else None

    # Heuristics: file sospetti
    suspicious = (
        (clean_words < 200) or               # troppo corto per un film
        (shrink_ratio is not None and shrink_ratio < 0.10)  # hai buttato via >90%
    )

    report.append({
        "imdb_id": imdb_id,
        "status": "ok",
        "raw_chars": raw_chars,
        "clean_chars": clean_chars,
        "raw_words": raw_words,
        "clean_words": clean_words,
        "shrink_ratio": shrink_ratio,
        "suspicious": suspicious
    })

report_df = pd.DataFrame(report)
report_path = LOGS_DIR / "subtitles_clean_overwrite_report.csv"
report_df.to_csv(report_path, index=False)

report_df["status"].value_counts(), report_df["suspicious"].value_counts(dropna=False)


Cleaning+overwrite: 100%|██████████| 3788/3788 [4:35:07<00:00,  4.36s/it]


(status
 ok    3788
 Name: count, dtype: int64,
 suspicious
 False    3776
 True       12
 Name: count, dtype: int64)

In [10]:
report_df = pd.read_csv("logs/subtitles_clean_overwrite_report.csv")
report_df.head()


,imdb_id,status,raw_chars,clean_chars,raw_words,clean_words,shrink_ratio,suspicious
0,tt0080684,ok,80994,40169,12123,7341,0.605543,False
1,tt0081562,ok,76054,40981,12016,7642,0.635985,False
2,tt0079417,ok,98216,49920,15201,9473,0.623183,False
3,tt0080339,ok,86002,44337,13040,8188,0.627914,False
4,tt0080377,ok,63229,31989,9970,6177,0.619559,False


In [13]:
sus = report_df.query("status=='ok' and suspicious==True").copy()
print("Sospetti:", len(sus), "su", (report_df["status"]=="ok").sum())

sus.sort_values(["clean_words"]).head(20)[["imdb_id","raw_words","clean_words","shrink_ratio"]]


Sospetti: 12 su 3788


,imdb_id,raw_words,clean_words,shrink_ratio
623,tt1071322,32,13,0.406250
2507,tt33397756,32,13,0.406250
3088,tt30870265,32,13,0.406250
3320,tt4836716,32,13,0.406250
3622,tt11464830,32,13,0.406250
2060,tt0251160,79,55,0.696203
3750,tt7134096,142,74,0.521127
3033,tt0485985,158,90,0.569620
2482,tt0408985,210,110,0.523810
3494,tt2239822,267,112,0.419476


In [16]:
test_id = "tt0020205"

srt_path = SUBS_RAW_DIR / f"{test_id}.srt"
txt_path = SUBS_CLEAN_DIR / f"{test_id}.txt"

print("SRT exists:", srt_path.exists(), "| TXT exists:", txt_path.exists())

raw = srt_path.read_text(encoding="utf-8", errors="ignore")
clean = txt_path.read_text(encoding="utf-8", errors="ignore")

print("\n--- RAW (prime 60 righe) ---")
print("\n".join(raw.splitlines()[:60]))

print("\n--- CLEAN (prime 60 righe) ---")
print("\n".join(clean.splitlines()[:60]))

print("\nraw_words:", len(raw.split()), "clean_words:", len(clean.split()))


SRT exists: True | TXT exists: True

--- RAW (prime 60 righe) ---
1
00:01:15,316 --> 00:01:17,855
THE LAST DAYS OF NAPOLEON
ON ST. HELENA

2
00:01:20,065 --> 00:01:22,692
Story and Scenario

3
00:01:24,833 --> 00:01:26,858
Direction

4
00:01:48,597 --> 00:01:51,704
Exterior views shot on St. Helena

5
00:01:54,721 --> 00:01:58,162
Cast

6
00:02:10,772 --> 00:02:16,570
June 1815. After Waterloo,
all was lost for Napoleon.

7
00:02:17,520 --> 00:02:20,153
Renounced by those who should
have defended him to the end,

8
00:02:20,192 --> 00:02:23,920
the former master of Europe
decided to abdicate.

9
00:02:41,697 --> 00:02:45,704
I didn't return from Elba so Paris
would be inundated with blood.

10
00:03:23,860 --> 00:03:28,323
While the cannon sounded at St. Denis
and the enemy was at the gates of Paris,

11
00:03:29,055 --> 00:03:32,376
Louis XVIII took possession
of the throne of France.

12
00:04:23,953 --> 00:04:26,533
Long live the King!

13
00:04:33,712 --> 00:04:40,007
December 7, 1

In [19]:
ready_df = report_df.query("status=='ok' and clean_words>=200 and shrink_ratio>=0.10").copy()
discard_df = report_df.query("status!='ok' or clean_words<200 or shrink_ratio<0.10").copy()

print("Pronti per LLM:", len(ready_df))
print("Da scartare/verificare:", len(discard_df))

ready_path = LOGS_DIR / "subtitles_ready_for_llm.csv"
discard_path = LOGS_DIR / "subtitles_discard_or_check.csv"

ready_df.to_csv(ready_path, index=False)
discard_df.to_csv(discard_path, index=False)

ready_df.head()


Pronti per LLM: 3776
Da scartare/verificare: 12


,imdb_id,status,raw_chars,clean_chars,raw_words,clean_words,shrink_ratio,suspicious
0,tt0080684,ok,80994,40169,12123,7341,0.605543,False
1,tt0081562,ok,76054,40981,12016,7642,0.635985,False
2,tt0079417,ok,98216,49920,15201,9473,0.623183,False
3,tt0080339,ok,86002,44337,13040,8188,0.627914,False
4,tt0080377,ok,63229,31989,9970,6177,0.619559,False


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=e08cfdf8-9b6e-44e8-b36c-3dd235d85ba1' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>